#### ***Setup (Run Before Attempting Questions) :***

Use the following two fine-tuned sequence classification checkpoints:

**DeBERTa:** microsoft/deberta-v3-small (fine-tuned checkpoint)

**RoBERTa:** roberta-base (fine-tuned checkpoint)    
**Label Mapping**

The models output logits for five labels corresponding to the answer options:

**Label ID**-----------**Option**

0---------------------A  

1---------------------B  

2---------------------C  

3---------------------D  

4---------------------E

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

df = pd.read_csv('train.csv')

deberta_model_name = "microsoft/deberta-v3-small"
roberta_model_name = "roberta-base"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_deberta = AutoTokenizer.from_pretrained(deberta_model_name)
model_deberta = AutoModelForSequenceClassification.from_pretrained(deberta_model_name, num_labels=5, ignore_mismatched_sizes=True).to(device)

tokenizer_roberta = AutoTokenizer.from_pretrained(roberta_model_name)
model_roberta = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=5, ignore_mismatched_sizes=True).to(device)

label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
print("Models reloaded with 5-label configuration.")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models reloaded with 5-label configuration.


#### **Load the fine-tuned DeBERTa and RoBERTa models.**

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

**Question 1:**

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?  
*(answer format : eg - A, probability of A)*

In [ ]:
def get_probs(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return softmax(logits, dim=1).cpu().numpy()[0]

# Row index 25 re-evaluation
row_25 = df.iloc[25]
prompt_25 = row_25['prompt']

probs_deberta_25 = get_probs(prompt_25, tokenizer_deberta, model_deberta)
highest_idx = np.argmax(probs_deberta_25)
print(f"Question 1: {label_map[highest_idx]}, {probs_deberta_25[highest_idx]:.4f}")

Question 1: D, 0.2539


#### **Using the same sample (row index 25), average the class probabilities from both models.**

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

**Question 2:**

Which answer option receives the highest averaged probability after simple probability ensembling?

In [ ]:
probs_roberta_25 = get_probs(prompt_25, tokenizer_roberta, model_roberta)
avg_probs_25 = (probs_deberta_25 + probs_roberta_25) / 2
highest_idx_avg = np.argmax(avg_probs_25)
print(f"Question 2: {label_map[highest_idx_avg]}")

Question 2: D


#### Apply **weighted probability averaging after Softmax** using the following weights:

**DeBERTa:** 0.70

**RoBERTa:** 0.30

**Compute**

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

**Question 3:**

Which answer option is ranked first after weighted ensembling?

In [ ]:
weighted_probs_25 = (0.7 * probs_deberta_25) + (0.3 * probs_roberta_25)
highest_idx_weighted = np.argmax(weighted_probs_25)
print(f"Question 3: {label_map[highest_idx_weighted]}")

Question 3: D


#### Using the weighted ensemble probabilities from **Q3**, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

**Question 4:**

What is the Top-3 prediction string for row index 25?  
**Example** : C A E

In [ ]:
top_3_indices = np.argsort(weighted_probs_25)[::-1][:3]
top_3_str = " ".join([label_map[i] for i in top_3_indices])
print(f"Question 4: {top_3_str}")

Question 4: D A C


#### Run the weighted ensemble pipeline on every row of **test.csv**.

Save the predictions in a file named **submission.csv** using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

**Question 5:**

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [ ]:
import csv

def get_top_3(probs):
    top_3 = np.argsort(probs)[::-1][:3]
    return " ".join([label_map[i] for i in top_3])

# Q5 specifically using the clarified dataframe
results = []
for idx, row in df.iterrows():
    p_deb = get_probs(row['prompt'], tokenizer_deberta, model_deberta)
    p_rob = get_probs(row['prompt'], tokenizer_roberta, model_roberta)
    p_final = (0.7 * p_deb) + (0.3 * p_rob)
    results.append({'id': row['id'], 'prediction': get_top_3(p_final)})

submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)
print(f"Question 5: {len(submission_df)}")

Question 5: 2000


#### For the first **50 rows of test.csv**, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using **DeBERTa** on both versions.

Average the predicted probabilities from both passes.

**Question 6:**

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [ ]:
df = pd.read_csv('test.csv')

diff_count_tta = 0
for idx in range(50):
    row = df.iloc[idx]
    prompt_orig = row['prompt']
    prompt_aug = "Answer the following multiple-choice question carefully: " + prompt_orig

    prob_orig = get_probs(prompt_orig, tokenizer_deberta, model_deberta)
    prob_aug = get_probs(prompt_aug, tokenizer_deberta, model_deberta)

    prob_tta = (prob_orig + prob_aug) / 2

    if np.argmax(prob_orig) != np.argmax(prob_tta):
        diff_count_tta += 1

print(f"Question 6: {diff_count_tta}")

Question 6: 0


#### Process the first **100 rows of test.csv**. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

**Question 7:**

How many rows have different Top-1 predictions?

In [ ]:
diff_top1_ensemble = 0
for idx in range(100):
    row = df.iloc[idx]
    p_deb = get_probs(row['prompt'], tokenizer_deberta, model_deberta)
    p_rob = get_probs(row['prompt'], tokenizer_roberta, model_roberta)
    p_ens = (0.7 * p_deb) + (0.3 * p_rob)

    if np.argmax(p_deb) != np.argmax(p_ens):
        diff_top1_ensemble += 1

print(f"Question 7: {diff_top1_ensemble}")

Question 7: 0


#### For the first **100 rows of test.csv**, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

**Question 8:**

How many rows have a positive confidence gain (greater than 0)?

In [ ]:
pos_gain_count = 0
for idx in range(100):
    row = df.iloc[idx]
    p_deb = get_probs(row['prompt'], tokenizer_deberta, model_deberta)
    p_rob = get_probs(row['prompt'], tokenizer_roberta, model_roberta)
    p_ens = (0.7 * p_deb) + (0.3 * p_rob)

    conf_deb = np.max(p_deb)
    conf_ens = np.max(p_ens)

    if (conf_ens - conf_deb) > 0:
        pos_gain_count += 1

print(f"Question 8: {pos_gain_count}")

Question 8: 23


#### For the **first 100 rows of test.csv**, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

**Question 9:**

How many rows have at least one change in their ordered Top-3 ranking after ensembling?  
Examples:  
A C D vs. A D C

In [ ]:
rank_change_count = 0
for idx in range(100):
    row = df.iloc[idx]
    p_deb = get_probs(row['prompt'], tokenizer_deberta, model_deberta)
    p_rob = get_probs(row['prompt'], tokenizer_roberta, model_roberta)
    p_ens = (0.7 * p_deb) + (0.3 * p_rob)

    top3_deb = get_top_3(p_deb)
    top3_ens = get_top_3(p_ens)

    if top3_deb != top3_ens:
        rank_change_count += 1

print(f"Question 9: {rank_change_count}")

Question 9: 0


#### Using the Top-3 predictions generated by your weighted ensemble for the **first 100 validation samples**, compute the MAP@3 score.

**Question 10:**

What is the final MAP@3 score?

In [ ]:
# Question 10 re-evaluation
def map_at_3(predictions, labels):
    scores = []
    for pred, label in zip(predictions, labels):
        pred_list = pred.split()
        if label == pred_list[0]:
            scores.append(1.0)
        elif len(pred_list) > 1 and label == pred_list[1]:
            scores.append(0.5)
        elif len(pred_list) > 2 and label == pred_list[2]:
            scores.append(0.3333)
        else:
            scores.append(0.0)
    return np.mean(scores)

val_df = pd.read_csv('train.csv').head(100)
val_preds = []
for idx, row in val_df.iterrows():
    p_deb = get_probs(row['prompt'], tokenizer_deberta, model_deberta)
    p_rob = get_probs(row['prompt'], tokenizer_roberta, model_roberta)
    p_ens = (0.7 * p_deb) + (0.3 * p_rob)
    val_preds.append(get_top_3(p_ens))

map_score = map_at_3(val_preds, val_df['answer'].values)
print(f"Question 10: {map_score:.4f}")

Question 10: 0.3733
